# Plot sea-ice changes #

In [1]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import pandas as pd
import xarray as xr
import numpy as np
from dask.distributed import Client
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.append('/home/563/sc1326/repos/cdrmip_extremes')
from cdrmip_extremes.configs import data_dir, models, expts
from cdrmip_extremes import load_esgf, utils, load_data
sys.path.append('/home/563/sc1326/Functions/')
from NCI_CMIP6_search import find_CMIP_files

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
up_df = find_CMIP_files(
    'CMIP',
    '1pctCO2',
    'SImon',
    'siconc'
).query(
    'source_id in @models & grid_label == "gn"'
).sort_values(by='source_id')
up_df

,activity_id,experiment_id,institution_id,source_id,member_id,table_id,variable_id,grid_label,version,path
48,CMIP,1pctCO2,CSIRO,ACCESS-ESM1-5,r1i1p1f1,SImon,siconc,gn,v20200817,[/g/data/fs38/publications/CMIP6/CMIP/CSIRO/AC...
10,CMIP,1pctCO2,NCAR,CESM2,r1i1p1f1,SImon,siconc,gn,v20190425,[/g/data/oi10/replicas/CMIP6/CMIP/NCAR/CESM2/1...
19,CMIP,1pctCO2,CNRM-CERFACS,CNRM-ESM2-1,r1i1p1f2,SImon,siconc,gn,v20181018,[/g/data/oi10/replicas/CMIP6/CMIP/CNRM-CERFACS...
39,CMIP,1pctCO2,CCCma,CanESM5,r1i1p1f1,SImon,siconc,gn,v20190429,[/g/data/oi10/replicas/CMIP6/CMIP/CCCma/CanESM...
2,CMIP,1pctCO2,NOAA-GFDL,GFDL-ESM4,r1i1p1f1,SImon,siconc,gn,v20180701,[/g/data/oi10/replicas/CMIP6/CMIP/NOAA-GFDL/GF...
35,CMIP,1pctCO2,MIROC,MIROC-ES2L,r1i1p1f2,SImon,siconc,gn,v20190823,[/g/data/oi10/replicas/CMIP6/CMIP/MIROC/MIROC-...
40,CMIP,1pctCO2,NCC,NorESM2-LM,r1i1p1f1,SImon,siconc,gn,v20190815,[/g/data/oi10/replicas/CMIP6/CMIP/NCC/NorESM2-...
22,CMIP,1pctCO2,MOHC,UKESM1-0-LL,r1i1p1f2,SImon,siconc,gn,v20200626,[/g/data/oi10/replicas/CMIP6/CMIP/MOHC/UKESM1-...


In [4]:
si_data = {model:{} for model in models}
for index, row in up_df.iterrows():
    model = row['source_id']
    path = row['path']
    print(f"loading {path}")
    si_data[model]['1pctCO2'] = xr.open_mfdataset(
        path,
        combine='nested',
        concat_dim='time',
        parallel=True,
        use_cftime=True,
        chunks={'time':100},
    ).siconc

loading ['/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20200817/siconc_SImon_ACCESS-ESM1-5_1pctCO2_r1i1p1f1_gn_010101-025012.nc']
loading ['/g/data/oi10/replicas/CMIP6/CMIP/NCAR/CESM2/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20190425/siconc_SImon_CESM2_1pctCO2_r1i1p1f1_gn_000101-005012.nc', '/g/data/oi10/replicas/CMIP6/CMIP/NCAR/CESM2/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20190425/siconc_SImon_CESM2_1pctCO2_r1i1p1f1_gn_005101-010012.nc', '/g/data/oi10/replicas/CMIP6/CMIP/NCAR/CESM2/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20190425/siconc_SImon_CESM2_1pctCO2_r1i1p1f1_gn_010101-015012.nc']


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'siconc' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'siconc' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'siconc' has multiple fill values {1e+20, 1e+20} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


loading ['/g/data/oi10/replicas/CMIP6/CMIP/CNRM-CERFACS/CNRM-ESM2-1/1pctCO2/r1i1p1f2/SImon/siconc/gn/v20181018/siconc_SImon_CNRM-ESM2-1_1pctCO2_r1i1p1f2_gn_185001-199912.nc']
loading ['/g/data/oi10/replicas/CMIP6/CMIP/CCCma/CanESM5/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20190429/siconc_SImon_CanESM5_1pctCO2_r1i1p1f1_gn_185001-200012.nc']
loading ['/g/data/oi10/replicas/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20180701/siconc_SImon_GFDL-ESM4_1pctCO2_r1i1p1f1_gn_000101-010012.nc', '/g/data/oi10/replicas/CMIP6/CMIP/NOAA-GFDL/GFDL-ESM4/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20180701/siconc_SImon_GFDL-ESM4_1pctCO2_r1i1p1f1_gn_010101-015012.nc']
loading ['/g/data/oi10/replicas/CMIP6/CMIP/MIROC/MIROC-ES2L/1pctCO2/r1i1p1f2/SImon/siconc/gn/v20190823/siconc_SImon_MIROC-ES2L_1pctCO2_r1i1p1f2_gn_185001-199912.nc']
loading ['/g/data/oi10/replicas/CMIP6/CMIP/NCC/NorESM2-LM/1pctCO2/r1i1p1f1/SImon/siconc/gn/v20190815/siconc_SImon_NorESM2-LM_1pctCO2_r1i1p1f1_gn_000101-001012.nc', '/g/data

In [5]:
si_data

{'ACCESS-ESM1-5': {'1pctCO2': <xarray.DataArray 'siconc' (time: 1800, j: 300, i: 360)> Size: 778MB
  dask.array<open_dataset-siconc, shape=(1800, 300, 360), dtype=float32, chunksize=(100, 300, 360), chunktype=numpy.ndarray>
  Coordinates:
    * time       (time) object 14kB 0101-01-16 12:00:00 ... 0250-12-16 12:00:00
    * j          (j) int32 1kB 0 1 2 3 4 5 6 7 ... 292 293 294 295 296 297 298 299
    * i          (i) int32 1kB 0 1 2 3 4 5 6 7 ... 352 353 354 355 356 357 358 359
      latitude   (j, i) float64 864kB dask.array<chunksize=(300, 360), meta=np.ndarray>
      longitude  (j, i) float64 864kB dask.array<chunksize=(300, 360), meta=np.ndarray>
      type       |S7 7B ...
  Attributes:
      standard_name:   sea_ice_area_fraction
      long_name:       Sea-Ice Area Percentage (Ocean Grid)
      comment:         Percentage of grid cell covered by sea ice
      units:           %
      original_units:  1
      history:         2020-08-17T00:29:09Z altered by CMOR: Converted units

In [6]:
# save
save_dir = os.path.join(data_dir,'raw/siconc/1pctCO2')
for model, ds_dict in si_data.items():
    if model not in ['UKESM1-0-LL']:
        continue
    ds = ds_dict['1pctCO2']
    path = os.path.join(
        save_dir,
        f"{model}_1pctCO2_siconc.nc"
    )
    ds.to_netcdf(path)